In [1]:
import sys
from pathlib import Path

print(sys.executable)
print(Path.cwd())

c:\temp\python_learning\ml_projects\ml_projects_batch_01\02_telco_customer_churn\.venv\Scripts\python.exe
c:\temp\python_learning\ml_projects\ml_projects_batch_01\02_telco_customer_churn\notebooks


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [3]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.shape

(7043, 21)

In [5]:
target_col = "Churn"

y = df[target_col].map({
    "No": 0,
    "Yes": 1,
})

y.head()

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64

In [6]:
y.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [7]:
y.value_counts(normalize=True)

Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

In [8]:
assert y.isna().sum() == 0
assert set(y.unique()) == {0, 1}

Ячейка 5 — create X

In [9]:
id_cols = ["customerID"]

X = df.drop(columns=id_cols + [target_col])

X.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


In [10]:
X.shape

(7043, 19)

In [11]:
assert "customerID" not in X.columns
assert "Churn" not in X.columns

Ячейка 6 — convert TotalCharges to numeric

In [12]:
X["TotalCharges"] = pd.to_numeric(X["TotalCharges"], errors="coerce")

In [13]:
X[["tenure", "MonthlyCharges", "TotalCharges"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tenure          7043 non-null   int64  
 1   MonthlyCharges  7043 non-null   float64
 2   TotalCharges    7032 non-null   float64
dtypes: float64(2), int64(1)
memory usage: 165.2 KB


In [14]:
X["TotalCharges"].isna().sum()

np.int64(11)

Ячейка 7 — define feature lists

In [15]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]

categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

In [16]:
all_features = numeric_features + categorical_features

len(numeric_features), len(categorical_features), len(all_features), X.shape[1]

(4, 15, 19, 19)

In [17]:
set(all_features) == set(X.columns)

True

In [18]:
assert set(all_features) == set(X.columns)

Ячейка 8 — stratified train/test split

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [20]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((5634, 19), (1409, 19), (5634,), (1409,))

Ячейка 9 — target distribution in full/train/test

In [21]:
split_distribution = pd.DataFrame({
    "full": y.value_counts(normalize=True).sort_index(),
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
})

split_distribution

,full,train,test
Churn,,,
0,0.73463,0.734647,0.734564
1,0.26537,0.265353,0.265436


In [22]:
split_counts = pd.DataFrame({
    "full": y.value_counts().sort_index(),
    "train": y_train.value_counts().sort_index(),
    "test": y_test.value_counts().sort_index(),
})

split_counts

,full,train,test
Churn,,,
0,5174,4139,1035
1,1869,1495,374


Ячейка 10 — preprocessing plan

## Stage 2 preprocessing plan

### Target

- Target column: `Churn`.
- Target encoding:
  - `No` -> 0
  - `Yes` -> 1
- Positive class: `1`, meaning `Churn = Yes`.

### Feature matrix

- `customerID` is dropped because it is an identifier.
- `Churn` is dropped from `X` because it is the target.
- `TotalCharges` is converted to numeric with `errors="coerce"`.
- `TotalCharges` missing values are not imputed at this stage.

### Feature groups

Numeric features:

- `SeniorCitizen`
- `tenure`
- `MonthlyCharges`
- `TotalCharges`

Categorical features:

- `gender`
- `Partner`
- `Dependents`
- `PhoneService`
- `MultipleLines`
- `InternetService`
- `OnlineSecurity`
- `OnlineBackup`
- `DeviceProtection`
- `TechSupport`
- `StreamingTV`
- `StreamingMovies`
- `Contract`
- `PaperlessBilling`
- `PaymentMethod`

### Train/test split

- `test_size=0.2`
- `random_state=42`
- `stratify=y`
- Stratification is used to preserve the churn class distribution in full, train, and test datasets.

### Future preprocessing plan

Numeric pipeline:

- `SimpleImputer`
  - likely strategy: `median`
- optional scaler:
  - useful for distance-based or linear models;
  - examples: KNN, Logistic Regression, Linear SVM;
  - less important for tree-based models.

Categorical pipeline:

- `SimpleImputer`
  - likely strategy: `most_frequent`
- `OneHotEncoder(handle_unknown="ignore")`
  - needed to safely handle categories in validation/test/inference that were not seen during fitting.

Combined preprocessing:

- numeric and categorical pipelines should be combined with `ColumnTransformer`.
- `ColumnTransformer` should be placed inside a full sklearn `Pipeline` together with the model.

### Leakage control

- Do not fit `SimpleImputer`, scaler, or `OneHotEncoder` on the full dataset.
- Do not use `pd.get_dummies()` on the full dataset as final training logic.
- Anything that learns parameters from data must be fitted only on `X_train` or inside cross-validation.
- The test set must not be used for model selection, threshold tuning, preprocessing decisions, or EDA-driven iteration.
- No models are trained at Stage 2.